# 02 — DistilBERT Rating-Sentiment Model

This notebook retrains and evaluates DistilBERT on the corrected UTC Amazon Beauty manifest. It uses the same 999,999 training IDs and evaluation IDs as the new TF-IDF `1m_v3` baseline; historical models remain untouched.

## Objectives

- train DistilBERT on 333,333 reviews from each rating-derived class;
- choose the best saved training point using validation data only;
- evaluate products excluded from training, later reviews, and `Conditioners`;
- compare DistilBERT and TF-IDF on identical test reviews;
- save the model, predictions, training details, and evaluation report.

# Модель тональности DistilBERT

Этот ноутбук заново обучает и проверяет DistilBERT на исправленном UTC-manifest Amazon Beauty. Модель использует те же 999 999 обучающих ID и evaluation ID, что и новая TF-IDF `1m_v3`; исторические модели не изменяются.

## Цели

- обучить DistilBERT на 333 333 отзывах каждого класса, полученного из рейтинга;
- выбрать лучшую сохраненную точку обучения только по отдельным данным для настройки;
- проверить товары, исключенные из обучения, более поздние отзывы и `Conditioners`;
- сравнить DistilBERT и TF-IDF на одних и тех же проверочных отзывах;
- сохранить модель, предсказания, сведения об обучении и отчет.

## 1. Inputs and label limitation

The notebook uses canonical reviews, the complete product catalog, and a saved table that assigns each `review_id` to one experiment part. Ratings 1–2 define `negative`, rating 3 defines `neutral`, and ratings 4–5 define `positive`.

These are imperfect labels. A customer can write positive text and accidentally select one star, write a mixed review, react emotionally, or rate delivery rather than the product. Therefore the measured result is agreement with rating-derived labels—not perfect understanding of human opinion. Such disagreements will be examined in the next error-analysis notebook.

## Входные данные и ограничение меток

Ноутбук использует очищенные отзывы, полный каталог товаров и сохраненную таблицу, которая относит каждый `review_id` к определенной части эксперимента. Оценки 1–2 означают `negative` («негативный»), оценка 3 — `neutral` («нейтральный»), оценки 4–5 — `positive` («позитивный»).

Эти метки неидеальны. Покупатель может написать положительный текст и случайно поставить одну звезду, смешать плюсы и минусы, написать отзыв на эмоциях или оценить доставку вместо товара. Поэтому результат показывает совпадение с метками, полученными из звезд, а не безошибочное понимание мнения человека. Такие случаи будут отдельно рассмотрены в следующем ноутбуке анализа ошибок.

In [ ]:
# Standard library / Стандартная библиотека
import json
import platform
import sys
from pathlib import Path

# Third-party packages / Сторонние библиотеки
import datasets
import joblib
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import torch
import transformers

# Locate the repository before importing reusable project code.
# Находим репозиторий до импорта переиспользуемого кода проекта.
PROJECT_ROOT = next(
    candidate
    for candidate in (Path.cwd(), *Path.cwd().parents)
    if (candidate / "PLAN.md").is_file() and (candidate / "src").is_dir()
)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Local project modules / Локальные модули проекта
from src.ingestion.dataset_manifest import load_dataset_manifest
from src.ml.sentiment import (
    SENTIMENT_LABELS,
    SentimentSplitConfig,
    audit_sentiment_sample,
    balanced_evaluation_view,
    build_sentiment_sample,
    evaluate_predictions,
    load_sentiment_split_reviews,
    predict_sentiment,
    write_sentiment_split_manifest,
)
from src.ml.transformer_sentiment import (
    TransformerSentimentConfig,
    predict_with_transformer,
    save_transformer_evaluation,
    train_or_load_transformer,
)

In [ ]:
# Corrected UTC lineage requires a new immutable model version.
# Исправленный UTC-lineage требует новой неизменяемой версии модели.
DATASET_VERSION = "amazon_reviews_2023_beauty_2021_2023_v1"
SPLIT_CONFIG = SentimentSplitConfig(
    split_version="beauty_rating_sentiment_split_1m_v3",
    train_per_class=333_333,
)
MODEL_CONFIG = TransformerSentimentConfig(
    model_version="beauty_distilbert_rating_sentiment_1m_v2",
)
MANIFEST_PATH = (
    PROJECT_ROOT / "config/datasets" / f"{DATASET_VERSION}.json"
)
DATASET_DIRECTORY = PROJECT_ROOT / "data/processed" / DATASET_VERSION
SPLIT_MANIFEST_PATH = (
    DATASET_DIRECTORY / "ml_splits" / f"{SPLIT_CONFIG.split_version}.parquet"
)
MODEL_DIRECTORY = PROJECT_ROOT / "models/sentiment-transformer"
CHECKPOINT_DIRECTORY = (
    PROJECT_ROOT / "models/sentiment-transformer-checkpoints"
)
PREDICTIONS_PATH = (
    DATASET_DIRECTORY
    / "model_predictions"
    / f"{MODEL_CONFIG.model_version}.parquet"
)
REPORT_PATH = (
    PROJECT_ROOT
    / "reports/model_evaluation"
    / f"{MODEL_CONFIG.model_version}.json"
)
TFIDF_MODEL_VERSION = "beauty_tfidf_rating_sentiment_1m_v3"
TFIDF_MODEL_DIRECTORY = PROJECT_ROOT / "models/sentiment-baseline"

display(
    pd.Series(
        {
            "dataset_version": DATASET_VERSION,
            "split_version": SPLIT_CONFIG.split_version,
            "training_reviews": SPLIT_CONFIG.train_per_class * 3,
            "model_version": MODEL_CONFIG.model_version,
            "source_model": MODEL_CONFIG.pretrained_model,
            "maximum_tokens_per_review": MODEL_CONFIG.max_length,
        },
        name="value",
    ).to_frame()
)

## 2. Environment and source files

Training requires CUDA and records the exact pretrained DistilBERT revision. The versioned model directory is never confused with the historical model. If a long run is interrupted, the latest saved training point can be continued.

## Окружение и исходные файлы

Для обучения требуется CUDA; также фиксируется точная версия исходной DistilBERT-модели. Папка новой модели не смешивается со старой исторической моделью. Если долгий запуск прервется, обучение можно продолжить с последней сохраненной точки.

In [ ]:
manifest = load_dataset_manifest(MANIFEST_PATH)
REVIEWS_PATH = PROJECT_ROOT / manifest.file_by_role("canonical_reviews").path
CATALOG_PATH = PROJECT_ROOT / manifest.file_by_role("product_catalog").path

assert manifest.dataset_version == DATASET_VERSION
assert REVIEWS_PATH.is_file()
assert CATALOG_PATH.is_file()
assert torch.cuda.is_available(), "DistilBERT training requires CUDA"

environment_summary = pd.Series(
    {
        "python": platform.python_version(),
        "python_executable": sys.executable,
        "torch": torch.__version__,
        "transformers": transformers.__version__,
        "datasets": datasets.__version__,
        "gpu": torch.cuda.get_device_name(0),
        "gpu_memory_gb": round(
            torch.cuda.get_device_properties(0).total_memory / 1024**3, 1
        ),
        "bfloat16_supported": torch.cuda.is_bf16_supported(),
    },
    name="value",
).to_frame()
display(environment_summary)

## 3. Fixed experiment parts

Training contains 333,333 examples of each class. Validation contains 20,000 examples of each class and is used to choose the best saved training point. The product tests preserve the real proportion of positive, neutral, and negative reviews. Products in validation and the two product tests are absent from training. Repeated normalized text is also absent across all parts.

## Зафиксированные части эксперимента

Обучение содержит по 333 333 примера каждого класса. Отдельная часть для настройки содержит по 20 000 примеров каждого класса и используется для выбора лучшей сохраненной точки обучения. Проверки по товарам сохраняют реальное соотношение положительных, нейтральных и негативных отзывов. Товары из настройки и двух товарных проверок отсутствуют в обучении. Одинаковые нормализованные тексты также не пересекаются между частями.

In [ ]:
if SPLIT_MANIFEST_PATH.is_file():
    split_manifest = pd.read_parquet(SPLIT_MANIFEST_PATH)
    split_was_created = False
else:
    complete_sample = build_sentiment_sample(
        REVIEWS_PATH,
        CATALOG_PATH,
        config=SPLIT_CONFIG,
    )
    write_sentiment_split_manifest(complete_sample, SPLIT_MANIFEST_PATH)
    split_manifest = complete_sample.drop(columns=["review_text"])
    split_was_created = True

split_distribution = (
    split_manifest.groupby(["split_name", "sentiment_label"])
    .size()
    .rename("review_count")
    .reset_index()
)
split_audit = audit_sentiment_sample(split_manifest)
split_audit_table = pd.DataFrame(split_audit["splits"])

assert split_audit["review_id_is_unique"]
assert split_audit["text_fingerprint_is_unique"]
assert len(split_manifest) == 1_269_020
assert str(split_manifest["review_timestamp"].dt.tz) == "UTC"
assert len(
    split_manifest.loc[split_manifest["split_name"] == "train"]
) == 999_999
assert (
    split_audit_table.loc[
        split_audit_table["split_name"].isin(
            ["validation", "test_product", "test_conditioners"]
        ),
        "train_product_overlap",
    ]
    == 0
).all()
cutoff = pd.Timestamp(SPLIT_CONFIG.temporal_cutoff, tz="UTC")
is_temporal = split_manifest["split_name"].eq("test_temporal_seen")
assert (split_manifest.loc[is_temporal, "review_timestamp"] >= cutoff).all()
assert (split_manifest.loc[~is_temporal, "review_timestamp"] < cutoff).all()

print(f"New split created: {split_was_created}")
display(split_distribution)
display(split_audit_table)

## 4. Train or load the versioned model

DistilBERT reads up to 256 tokens from each review and learns one class from the complete text. Training runs for one pass over 999,999 reviews. Every 5,000 steps the model is checked on separate validation data and saved; after training, the saved point with the best equal-class F1 result is restored. Dynamic padding avoids filling every short review to 256 tokens.

When the base model is loaded, the library reports that its new three-class output layer is initially empty. This is expected: general English weights are reused, while the sentiment output layer must be learned from our Amazon reviews.

## Обучение или загрузка модели

DistilBERT читает до 256 частей текста (токенов) из каждого отзыва и определяет один общий класс. Модель один раз проходит по 999 999 обучающим отзывам. Каждые 5 000 шагов она проверяется на отдельных данных и сохраняется; после обучения восстанавливается точка с лучшим показателем F1, где все классы имеют равный вес. Динамическое дополнение не увеличивает каждый короткий отзыв до 256 токенов и экономит вычисления.

При загрузке исходной модели библиотека сообщает, что новый выходной слой для трех классов сначала пуст. Это ожидаемо: общие знания английского языка берутся из готовой модели, а последний слой тональности обучается на наших Amazon-отзывах.

In [ ]:
training_reviews = load_sentiment_split_reviews(
    REVIEWS_PATH,
    SPLIT_MANIFEST_PATH,
    split_names=("train", "validation"),
)
train_reviews = training_reviews.loc[
    training_reviews["split_name"] == "train"
].copy()
validation_reviews = training_reviews.loc[
    training_reviews["split_name"] == "validation"
].copy()

model, tokenizer, trainer, training_summary = train_or_load_transformer(
    train_reviews,
    validation_reviews,
    config=MODEL_CONFIG,
    model_directory=MODEL_DIRECTORY,
    checkpoint_directory=CHECKPOINT_DIRECTORY,
)

assert (MODEL_DIRECTORY / "model.safetensors").is_file()
assert model.config.id2label == {
    index: label for index, label in enumerate(SENTIMENT_LABELS)
}
display(pd.Series(training_summary, name="value").to_frame())

## 5. Model evaluation

Three independent questions are measured. First: how well does the model work for Beauty products whose reviews were excluded from training? Second: how well does it handle later 2023 reviews for products represented in training? Third: how well does it work on held-out `Conditioners` parent products while other Conditioner products remain represented in training?

For each question, the natural result keeps the real class proportions. The balanced result uses the same number of examples from each class, so a large positive class cannot hide weak Neutral performance. The 95% interval resamples whole `parent_asin` clusters, keeping reviews of the same product together.

## Проверка качества модели

Измеряются три независимых результата. Первый: как модель работает с товарами Beauty, отзывы о которых не использовались при обучении? Второй: как она обрабатывает более поздние отзывы 2023 года о знакомых товарах? Третий: как она работает на held-out parent products из `Conditioners`, хотя другие товары Conditioner остаются представлены в обучении?

Для каждого вопроса естественный результат сохраняет реальное соотношение классов. Сбалансированный результат берет одинаковое количество примеров каждого класса, поэтому большой положительный класс не скрывает слабое распознавание Neutral. Интервал 95% пересэмплирует целые кластеры `parent_asin`, сохраняя отзывы одного товара вместе.

In [ ]:
evaluation_reviews = load_sentiment_split_reviews(
    REVIEWS_PATH,
    SPLIT_MANIFEST_PATH,
    split_names=(
        "validation",
        "test_product",
        "test_temporal_seen",
        "test_conditioners",
    ),
)
transformer_predictions = predict_with_transformer(
    evaluation_reviews,
    trainer=trainer,
    tokenizer=tokenizer,
    max_length=MODEL_CONFIG.max_length,
)

token_summary = pd.Series(
    {
        "evaluated_reviews": len(transformer_predictions),
        "median_tokens": transformer_predictions["token_count"].median(),
        "tokens_p90": transformer_predictions["token_count"].quantile(0.90),
        "tokens_p99": transformer_predictions["token_count"].quantile(0.99),
        "truncated_reviews": transformer_predictions["was_truncated"].sum(),
        "truncated_share_pct": (
            transformer_predictions["was_truncated"].mean() * 100
        ),
    },
    name="value",
).to_frame()
display(token_summary)

In [ ]:
evaluation_views = {
    "validation_balanced": transformer_predictions.loc[
        transformer_predictions["split_name"] == "validation"
    ]
}
for split_name in (
    "test_product",
    "test_temporal_seen",
    "test_conditioners",
):
    natural_view = transformer_predictions.loc[
        transformer_predictions["split_name"] == split_name
    ]
    evaluation_views[f"{split_name}_natural"] = natural_view
    evaluation_views[f"{split_name}_balanced"] = (
        balanced_evaluation_view(
            natural_view,
            max_per_class=20_000,
            random_state=MODEL_CONFIG.random_state,
        )
    )

transformer_metrics = {}
transformer_metric_rows = []
for view_name, view in evaluation_views.items():
    view_metrics = evaluate_predictions(
        view,
        bootstrap_rounds=200,
        random_state=MODEL_CONFIG.random_state,
    )
    transformer_metrics[view_name] = view_metrics
    transformer_metric_rows.append(
        {
            "evaluation_view": view_name,
            "reviews": view_metrics["review_count"],
            "accuracy": view_metrics["accuracy"],
            "macro_f1": view_metrics["macro_f1"],
            "weighted_f1": view_metrics["weighted_f1"],
            "neutral_f1": view_metrics["per_class"]["neutral"][
                "f1-score"
            ],
            "macro_f1_ci_low": view_metrics[
                "confidence_intervals_95"
            ]["macro_f1"][0],
            "macro_f1_ci_high": view_metrics[
                "confidence_intervals_95"
            ]["macro_f1"][1],
        }
    )

transformer_metrics_table = pd.DataFrame(transformer_metric_rows)
display(transformer_metrics_table.round(4))

## 6. Comparison with TF-IDF on identical data

The saved TF-IDF 1M model is run on the current evaluation rows. Both models were trained on the same 999,999 review IDs and answer the same validation and final-test reviews. The remaining difference therefore reflects the model methods and their training procedures, not the amount of training text.

## Сравнение с TF-IDF на одинаковых данных

Сохранённая TF-IDF на одном миллионе запускается на текущих проверочных строках. Обе модели обучались на одинаковых 999 999 `review_id` и отвечают на одинаковые строки настройки и итоговых проверок. Поэтому оставшаяся разница связана с методами моделей и способом их обучения, а не с количеством обучающего текста.

In [ ]:
tfidf_experiment = json.loads(
    (TFIDF_MODEL_DIRECTORY / "experiment_config.json").read_text(
        encoding="utf-8"
    )
)
assert tfidf_experiment["dataset_version"] == DATASET_VERSION
assert tfidf_experiment["split_config"]["split_version"] == (
    SPLIT_CONFIG.split_version
)
assert tfidf_experiment["model_config"]["model_version"] == (
    TFIDF_MODEL_VERSION
)
tfidf_vectorizer = joblib.load(TFIDF_MODEL_DIRECTORY / "vectorizer.joblib")
tfidf_classifier = joblib.load(TFIDF_MODEL_DIRECTORY / "classifier.joblib")
tfidf_predictions = predict_sentiment(
    evaluation_reviews,
    vectorizer=tfidf_vectorizer,
    classifier=tfidf_classifier,
)

tfidf_metrics = {}
comparison_rows = []
tfidf_by_review = tfidf_predictions.set_index("review_id")
for view_name, transformer_view in evaluation_views.items():
    tfidf_view = tfidf_by_review.loc[
        transformer_view["review_id"]
    ].reset_index()
    assert tfidf_view["review_id"].tolist() == transformer_view[
        "review_id"
    ].tolist()
    baseline_view_metrics = evaluate_predictions(
        tfidf_view,
        bootstrap_rounds=200,
        random_state=MODEL_CONFIG.random_state,
    )
    tfidf_metrics[view_name] = baseline_view_metrics
    for model_name, model_metrics in (
        ("TF-IDF (1m training reviews)", baseline_view_metrics),
        ("DistilBERT (1m training reviews)", transformer_metrics[view_name]),
    ):
        comparison_rows.append(
            {
                "evaluation_view": view_name,
                "model": model_name,
                "accuracy": model_metrics["accuracy"],
                "macro_f1": model_metrics["macro_f1"],
                "neutral_f1": model_metrics["per_class"]["neutral"][
                    "f1-score"
                ],
            }
        )

comparison_table = pd.DataFrame(comparison_rows)
display(comparison_table.round(4))

In [ ]:
# Show where each model confuses the three classes on the balanced product test.
# Показываем, какие классы путает каждая модель на сбалансированной проверке
# по товарам.
figure, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for axis, model_name, metric_source in (
    (axes[0], "TF-IDF", tfidf_metrics),
    (axes[1], "DistilBERT", transformer_metrics),
):
    matrix = metric_source["test_product_balanced"]["confusion_matrix"]
    sns.heatmap(
        matrix,
        annot=True,
        fmt=",",
        cmap="Blues",
        xticklabels=SENTIMENT_LABELS,
        yticklabels=SENTIMENT_LABELS,
        ax=axis,
    )
    axis.set_title(model_name)
    axis.set_xlabel("Predicted class")
    axis.set_ylabel("Class derived from stars")
figure.suptitle("Errors on products excluded from training")
figure.tight_layout()
plt.show()

## 7. Save and validate results

The final model is stored under a new version name. Predictions retain review and product identifiers, original token counts, the fact of text shortening, three class scores, and the selected label. The report also keeps training settings and both models' results.

## Сохранение и проверка результатов

Готовая модель сохраняется под новым номером версии. В таблице предсказаний остаются идентификаторы отзывов и товаров, исходное число токенов, признак сокращения длинного текста, оценки трех классов и выбранный класс. Отчет также хранит настройки обучения и результаты обеих моделей.

In [ ]:
saved_paths = save_transformer_evaluation(
    predictions=transformer_predictions,
    metrics={
        "distilbert": transformer_metrics,
        "tfidf_same_test_rows": tfidf_metrics,
    },
    training_summary=training_summary,
    config=MODEL_CONFIG,
    dataset_version=DATASET_VERSION,
    split_version=SPLIT_CONFIG.split_version,
    predictions_path=PREDICTIONS_PATH,
    report_path=REPORT_PATH,
)

assert (MODEL_DIRECTORY / "model.safetensors").is_file()
assert (MODEL_DIRECTORY / "tokenizer.json").is_file()
assert PREDICTIONS_PATH.is_file()
assert REPORT_PATH.is_file()
assert len(pd.read_parquet(PREDICTIONS_PATH)) == len(transformer_predictions)
display(pd.Series(saved_paths, name="path").to_frame())

## 8. Conclusion and next check

Both new models use the exact corrected `1m_v3` training and evaluation IDs. The tables above and the versioned JSON report contain the clean-run point estimates, product-cluster intervals, per-class metrics, and confusion matrices; no value is inherited from the superseded timezone-sensitive run.

TF-IDF remains the fast, understandable reference. The training summaries record the measured duration of both clean runs. Error Analysis checks individual failures, probability reliability, Neutral answers, text-rating disagreement, mixed opinions, delivery comments, and shortened long texts.

This DistilBERT version remains a candidate until its newly sampled review queue is labeled. Human labels from the historical model are not transferred to the new queue.

## Вывод и следующая проверка

Обе новые модели используют строго одинаковые исправленные train- и evaluation ID из `1m_v3`. Таблицы выше и версионированный JSON-отчёт содержат оценки clean-run, интервалы по product-cluster bootstrap, метрики по классам и confusion matrices; значения из заменённого timezone-зависимого запуска не переносятся.

TF-IDF остаётся быстрой и понятной reference-моделью. Реальная длительность обоих чистых запусков записывается в training summaries. Ноутбук Error Analysis проверяет отдельные ошибки, надёжность вероятностей, ответы Neutral, несовпадения текста и звёзд, смешанные мнения, замечания о доставке и сокращённые длинные отзывы.

Эта версия DistilBERT остаётся кандидатом до разметки её новой review queue. Человеческие метки исторической модели в новую очередь не переносятся.